# Clase 4 · 05 · Validación del experimento y reejecución
Los controles usan la secuencia de llegadas, no un número fijo de microbatches.
Verificamos salida, eventos aceptados/descartados, ventanas finalizadas e idempotencia.

In [ ]:
dbutils.widgets.text("student_id", "alumno01", "01 - Identificador")
dbutils.widgets.dropdown("scale", "small", ["test", "small", "demo"], "02 - Escala de clases 1 y 2")
dbutils.widgets.dropdown("expected_batch_id", "stream_001",
                         ["stream_001", "stream_002", "stream_003", "stream_004", "stream_005"], "03 - Etapa a validar")
dbutils.widgets.text("job_run_id", "interactive", "04 - ID de ejecución")

In [ ]:
%run ../common/config

In [ ]:
%run ../common/generate_stream_data

In [ ]:
%run ../common/streaming_support

In [ ]:
from pyspark.sql import functions as F
config = CourseConfig(spark.sql("SELECT current_catalog()").first()[0],
                      dbutils.widgets.get("student_id"), dbutils.widgets.get("scale"))
repetitions = STREAM_REPETITIONS[config.scale]
expected_batch_id = normalize_stream_batch(dbutils.widgets.get("expected_batch_id"))
job_run_id = dbutils.widgets.get("job_run_id").strip()[:100] or "interactive"
spark.conf.set("spark.sql.session.timeZone", "UTC")
require_stream_lab(spark, config)

In [ ]:
%run ../common/streaming_expectations

## 1. Reconciliar la etapa y registrar evidencia
No reconciliamos ventanas abiertas como si estuvieran finalizadas.
Los checkpoints determinan el trabajo pendiente; `expected_batch_id` sólo define la expectativa del control.
Una reejecución consecutiva de la misma etapa compara cantidades y firmas de contenido.

In [ ]:
from decimal import Decimal
import json
import uuid

published = published_stream_batches(dbutils, config)
expected_prefix = list(STREAM_BATCHES[:STREAM_BATCHES.index(expected_batch_id) + 1])
assert published == expected_prefix, f"La etapa espera {expected_prefix}, pero hay {published}. No publiques etapas juntas."
tables = {"bronze": "bronze_stream_events", "classified": "silver_stream_classified",
          "valid": "silver_stream_valid", "quarantine": "silver_stream_quarantine",
          "unique": "silver_stream_events", "gold": "gold_stream_windows"}
actual_counts = {key: spark.table(name).count() for key, name in tables.items()}
assert actual_counts == expected_stream_counts(expected_batch_id, repetitions), actual_counts
assert actual_counts["bronze"] == actual_counts["valid"] + actual_counts["quarantine"]
assert spark.table("silver_stream_events").select("event_id").distinct().count() == actual_counts["unique"]

expected_by_batch = {batch: len(build_stream_rows(config, batch)) for batch in expected_prefix}
actual_by_batch = {r.source_batch_id: r['count'] for r in
                  spark.table("silver_stream_classified").groupBy("source_batch_id").count().collect()}
assert actual_by_batch == expected_by_batch
reasons = {r.quality_reason: r['count'] for r in
           spark.table("silver_stream_quarantine").groupBy("quality_reason").count().collect()}
assert reasons == ({} if expected_batch_id == "stream_001"
                   else {"INVALID_AMOUNT": repetitions, "UNKNOWN_CUSTOMER": repetitions})
gold_rows = spark.table("gold_stream_windows").orderBy("window_start", "payment_channel").collect()
actual_gold = {(r.window_start.strftime("%Y-%m-%d %H:%M:%S"), r.payment_channel):
               (r.event_count, r.total_amount, r.fraud_count) for r in gold_rows}
assert len(actual_gold) == len(gold_rows), "Gold tiene claves duplicadas"
assert actual_gold == expected_gold_rows(expected_batch_id, repetitions), actual_gold
for row in gold_rows:
    assert (row.window_end - row.window_start).total_seconds() == 300
    assert row.payment_channel != "control"

if expected_batch_id >= "stream_002":
    assert spark.table("silver_stream_valid").where("event_id = 'e002_000'").count() == 2
    assert spark.table("silver_stream_events").where("event_id = 'e002_000'").count() == 1
    assert spark.table("silver_stream_events").where("event_id = 'late_ok_000'").count() == 1
if expected_batch_id >= "stream_004":
    assert spark.table("silver_stream_valid").where("event_id = 'late_bad_000'").count() == 1
    assert spark.table("silver_stream_events").where("event_id = 'late_bad_000'").count() == 0
if expected_batch_id == "stream_005":
    assert sum(r.event_count for r in gold_rows) == 9 * repetitions
    assert sum((r.total_amount for r in gold_rows), Decimal(0)) == Decimal("1275.00") * repetitions

# Una firma pequeña de contenidos, además de las cantidades. La auditoría y
# el progreso no se comparan: agregan evidencia nueva aunque no cambie negocio.
raw_hash = spark.table("bronze_stream_events").agg(
    F.sum(F.xxhash64("raw_json").cast("decimal(38,0)")).alias("value")).first().value
unique_hash = spark.table("silver_stream_events").agg(
    F.sum(F.xxhash64(*spark.table("silver_stream_events").columns).cast("decimal(38,0)")).alias("value")).first().value
signature = {"counts": actual_counts, "raw_hash": str(raw_hash), "unique_hash": str(unique_hash),
             "gold": [list(key) + [str(value) for value in values]
                      for key, values in sorted(actual_gold.items())]}
metrics_json = json.dumps(signature, sort_keys=True)
prior = spark.table("stream_run_audit").orderBy(F.col("recorded_at").desc()).first()
compared = prior is not None and prior.expected_batch_id == expected_batch_id
if compared:
    assert prior.metrics_json == metrics_json, "La reejecución cambió las métricas o contenidos de negocio"
validation_id = str(uuid.uuid4())
audit_row = (spark.createDataFrame([(validation_id, job_run_id, expected_batch_id, metrics_json, compared)],
             "validation_id STRING, job_run_id STRING, expected_batch_id STRING, metrics_json STRING, idempotence_compared BOOLEAN")
             .withColumn("recorded_at", F.current_timestamp()))
audit_row.write.format("delta").mode("append").saveAsTable(qualified_table(config, "stream_run_audit"))
display(spark.createDataFrame([(key, count) for key, count in actual_counts.items()], ["resource", "rows"]))
print("Validación completa:", expected_batch_id, "Reejecución comparada:", compared)

## 2. Revisar watermark y estado
El progreso puede incluir un microbatch sin entradas que avance la limpieza de estado y finalice ventanas.
Su cantidad y su batchId dependen del runtime; no son identidades de los archivos de entrada.
No interpretes tasas de procesamiento como un benchmark del TP.

In [ ]:
%sql
SELECT stage, get_json_object(progress_json, '$.batchId') AS microbatch_id,
       get_json_object(progress_json, '$.numInputRows') AS input_rows,
       get_json_object(progress_json, '$.eventTime.watermark') AS watermark,
       get_json_object(progress_json, '$.stateOperators') AS state_operators,
       recorded_at
FROM stream_query_progress ORDER BY recorded_at DESC LIMIT 20

In [ ]:
%sql
SELECT job_run_id, expected_batch_id, idempotence_compared, recorded_at
FROM stream_run_audit ORDER BY recorded_at DESC LIMIT 10

### Próxima acción
- Después de stream_001 y stream_005, repetí el mismo Job sin publicar otro archivo y conservá la evidencia.
- Después de validar una etapa, publicá la siguiente de la secuencia.
- Tras 005 y su reejecución, ejecutá 06_visualizacion y 07_desafio.